# Baseline: referência inicial no treino

Comparamos o Dummy (sempre prevê a classe majoritária) com regressão logística.
Usamos o mesmo `data/train.csv`, a mesma divisão estratificada 80/20 e a semente 42 do fluxo principal.
Todas as comparações abaixo usam validação cruzada somente nos 80% de treino.
O preenchimento de nulos, a codificação e a padronização são aprendidos dentro de cada rodada, pelo pipeline compartilhado.

**Histórico:** a versão anterior deste notebook consultou o teste e comparou limiares nele.
A correção evita novas consultas aqui, mas não transforma o teste já conhecido em uma avaliação independente.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Abra o notebook na raiz do projeto ou na pasta notebooks.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.baseline import run_baseline
from src.pipeline import SEED, read_passengers


## Separar antes de explorar

Não inspecionamos nem avaliamos os passageiros reservados para teste.
A avaliação final permanece em `src.train`, depois de fixar modelo, parâmetros e limiar.


In [ ]:
data = read_passengers(ROOT / "data/train.csv", training=True)
train, _ = train_test_split(
    data, test_size=0.2, stratify=data["Survived"], random_state=SEED,
)
display(train.head())
display(train.isna().sum())
display(train["Survived"].value_counts(normalize=True))


## Comparação em cinco partes do treino

Cada pipeline é ajustado apenas na parte de treinamento de cada rodada.
As colunas `cv_*` são médias de validação interna com o limiar padrão 0,50.
O Dummy pode ter precisão, recall e F1 iguais a zero porque não prevê sobreviventes.


In [ ]:
results = run_baseline(ROOT / "data/train.csv")
display(results["comparison"].round(4))
print(results["note"])


## Explorar limiares somente no treino

Cada probabilidade é produzida por um pipeline que não treinou com aquele passageiro.
Escolhemos o limiar pelo F1 dessas previsões de validação cruzada.
**O F1 usado para escolher o limiar é uma medida de seleção interna, não uma avaliação independente do limiar escolhido.**
O fluxo principal repete sua própria seleção; este notebook não salva nem substitui o modelo ou os relatórios finais.


In [ ]:
display(results["thresholds"].round(4))
print(f'Limiar escolhido na validação interna: {results["threshold"]:.2f}')
